# 03 - Text Preprocessing & Document Embeddings

**Objective**: Clean scientific text and generate SPECTER2 document embeddings.

**Input**: `outputs/mhealth_screened_corpus.csv`

**Outputs**:
- `outputs/mhealth_preprocessed.csv`
- `outputs/document_embeddings.npy`
- `outputs/document_embedding_index.csv`
- `outputs/embedding_model_info.txt`

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [2]:
import pandas as pd
from src.config_loader import load_config, set_seed, ensure_dirs
from src.text_preprocessing import preprocess_corpus, save_preprocessed
from src.embeddings import generate_embeddings

dirs = ensure_dirs()
set_seed()

[CONFIG] Random seed set to 42


## Load Screened Corpus

In [3]:
df_screened = pd.read_csv(
    dirs['outputs'] / 'mhealth_screened_corpus.csv',
    encoding='utf-8-sig',
)
print(f'Loaded {len(df_screened):,} screened records')

Loaded 12,642 screened records


## Text Preprocessing

In [4]:
df_processed = preprocess_corpus(df_screened)
save_preprocessed(df_processed)


[PREPROCESS] Cleaning text fields...
  Input records: 12,642


  Removed 0 records with clean_document < 50 chars
  Final preprocessed corpus: 12,642 records

--- Before/After Text Examples ---

  PMID 24282189:
    ORIGINAL: Developing requirements for a mobile app to support citizens in dealing with ticks and tick bites via end-user profiling.. Tick bites and tick-borne infections are an increasingly large problem. There...
    CLEANED:  Developing requirements for a mobile app to support citizens in dealing with ticks and tick bites via end user profiling.. Tick bites and tick borne infections are an increasingly large problem. There...

  PMID 24313728:
    ORIGINAL: Pilot feasibility study of a brief, tailored mobile health intervention for depression among patients with chronic pain.. This pilot feasibility study investigated a brief, tailored mobile health inte...
    CLEANED:  Pilot feasibility study of a brief, tailored mobile health intervention for depression among patients with chronic pain.. This pilot feasibility study investigated a

[SAVE] Preprocessed corpus -> E:\Major_Project_Implementation\outputs\mhealth_preprocessed.csv  (12,642 records)


{'preprocessed': WindowsPath('E:/Major_Project_Implementation/outputs/mhealth_preprocessed.csv')}

## Generate Document Embeddings

This step downloads the SPECTER2 model (~440MB) and encodes all documents.
Uses GPU if available. Supports resume if embeddings already exist.

In [5]:
embeddings, emb_index = generate_embeddings(
    df_processed,
    text_column='clean_document',
    resume=True,
)
print(f'\nEmbedding shape: {embeddings.shape}')
print(f'Index shape: {emb_index.shape}')

[EMBED] Found existing embeddings at E:\Major_Project_Implementation\outputs\document_embeddings.npy
[EMBED] Loaded 12642 embeddings, dim=768

Embedding shape: (12642, 768)
Index shape: (12642, 2)
